In [1]:
import datetime
import re
from concurrent.futures import ThreadPoolExecutor, as_completed
import bs4
import pandas as pd
import requests

# -------------------------------------------------------------------------
# Configuration
# -------------------------------------------------------------------------
budget_account_list = [9130]
fy_list = list(range(2002, 2027))  # 2002 to 2026

# Step 1: Get Level 1 IDs
level_1_rows = []

for ba in budget_account_list:
    for fy in fy_list:
        url = f"http://dawn12.state.nv.us{ba}&input_fund=101&input_fiscal_year={fy}"

        try:
            response = requests.get(url, timeout=15)
            response.raise_for_status()
            soup = bs4.BeautifulSoup(response.text, "html.parser")

            # Extract all anchor links
            links = [a.get("href") for a in soup.find_all("a") if a.get("href")]

            # Target links containing 'bsr_rec_fund_sum' (Revenues)
            target_links = [ln for ln in links if "bsr_rec_fund_sum" in ln]

            for link in target_links:
                # Extract numeric characters following 'level1_id='
                match = re.search(r"level1_id=(\d+)", link)
                if match:
                    level_1_id = match.group(1)
                    level_1_rows.append(
                        {"level_1": level_1_id, "ba": ba, "fy": fy}
                    )
        except Exception as e:
            print(f"Error processing Step 1 for BA {ba}, FY {fy}: {e}")

level_1_ids = pd.DataFrame(
    level_1_rows, columns=["level_1", "ba", "fy"]
).drop_duplicates()


# -------------------------------------------------------------------------
# Step 2: Given a level 1 ID, get a list of corresponding level 2 IDs
# -------------------------------------------------------------------------
level_2_rows = []

for index, row in level_1_ids.iterrows():
    l1_id = row["level_1"]
    ba = row["ba"]
    fy = row["fy"]

    url = f"http://dawn12.state.nv.us{l1_id}"

    try:
        response = requests.get(url, timeout=15)
        response.raise_for_status()
        soup = bs4.BeautifulSoup(response.text, "html.parser")

        # Find all anchor tags within table cells (td a)
        td_anchors = []
        for td in soup.find_all("td"):
            td_anchors.extend(td.find_all("a"))

        for anchor in td_anchors:
            href = anchor.get("href", "")
            revenue_account = anchor.get_text(strip=True)

            match = re.search(r"pm_level2_id=(\d+)", href)
            if match:
                level2_id = match.group(1)

                # Keep row if level2_id is fully numeric
                if level2_id.isdigit():
                    level_2_rows.append(
                        {
                            "level2_id": level2_id,
                            "revenue_account": revenue_account,
                            "level_1_id": str(l1_id),
                            "ba": int(ba),
                            "fy": int(fy),
                        }
                    )
    except Exception as e:
        print(f"Error processing Step 2 for level_1_id {l1_id}: {e}")

# Build DataFrame and filter for numeric revenue accounts (equivalent to R's filter(!is.na(as.numeric(revenue_account))))
level_2_ids = pd.DataFrame(level_2_rows)
if not level_2_ids.empty:
    level_2_ids = level_2_ids[level_2_ids["revenue_account"].str.isdigit()]


# -------------------------------------------------------------------------
# Step 3: Get transaction details concurrently
# -------------------------------------------------------------------------
to_date = datetime.date.today().strftime("%m/%d/%Y")


def download_transaction_data(level1_id, level2_id, category, ba, fy):
    url = (
        f"http://dawn12.state.nv.us?"
        f"pm_level1_id={level1_id}&pm_level2_id={level2_id}"
        f"&pm_level3_id=&pm_from_date=01/01/2000&pm_to_date={to_date}"
    )
    try:
        # pd.read_html can fetch data directly from a URL or raw HTML string.
        # R's htmltab(url, which = 4) maps to extracting the 4th table (index 3 in Python)
        dfs = pd.read_html(url)
        if len(dfs) >= 4:
            df = dfs[3]  # 4th table extraction
            df["level1_id"] = level1_id
            df["level2_id"] = level2_id
            df["category"] = category
            df["ba"] = ba
            df["fy"] = fy
            return df
        else:
            return None
    except Exception as e:
        print(
            f"Error downloading data for level1_id={level1_id} level2_id={level2_id}: {e}"
        )
        return None


# Filter for revenue account 3068 (same as GL 3068)
if not level_2_ids.empty:
    filtered_level_2 = level_2_ids[level_2_ids["revenue_account"] == "3068"]
else:
    filtered_level_2 = pd.DataFrame()

results = []

# Using ThreadPoolExecutor for concurrent requests (replaces R's future/plan)
with ThreadPoolExecutor(max_workers=8) as executor:
    futures = []
    for _, row in filtered_level_2.iterrows():
        futures.append(
            executor.submit(
                download_transaction_data,
                level1_id=row["level_1_id"],
                level2_id=row["level2_id"],
                category=row["revenue_account"],
                ba=row["ba"],
                fy=row["fy"],
            )
        )

    for future in as_completed(futures):
        res = future.result()
        if res is not None and not res.empty:
            results.append(res)

# Combine, clean, and export data
if results:
    result_df = pd.concat(results, ignore_index=True)

    # Filter out rows missing 'Doc Number'
    if "Doc Number" in result_df.columns:
        revenue_data = result_df[result_df["Doc Number"].notna()].copy()

        # Safely convert 'Date' column to Datetime format
        if "Date" in revenue_data.columns:
            revenue_data["Date"] = pd.to_datetime(
                revenue_data["Date"], errors="coerce"
            )

        # Export to CSV (corresponds to fwrite)
        revenue_data.to_csv("bank_exise_tax_history.csv", index=False)
        print("Scraping completed. File 'bank_exise_tax_history.csv' saved.")
else:
    print("No transaction records found matching the criteria.")


Error processing Step 1 for BA 9130, FY 2002: HTTPConnectionPool(host='dawn12.state.nv.us9130&input_fund=101&input_fiscal_year=2002', port=80): Max retries exceeded with url: / (Caused by NameResolutionError("HTTPConnection(host='dawn12.state.nv.us9130&input_fund=101&input_fiscal_year=2002', port=80): Failed to resolve 'dawn12.state.nv.us9130&input_fund=101&input_fiscal_year=2002' ([Errno 11001] getaddrinfo failed)"))
Error processing Step 1 for BA 9130, FY 2003: HTTPConnectionPool(host='dawn12.state.nv.us9130&input_fund=101&input_fiscal_year=2003', port=80): Max retries exceeded with url: / (Caused by NameResolutionError("HTTPConnection(host='dawn12.state.nv.us9130&input_fund=101&input_fiscal_year=2003', port=80): Failed to resolve 'dawn12.state.nv.us9130&input_fund=101&input_fiscal_year=2003' ([Errno 11001] getaddrinfo failed)"))
Error processing Step 1 for BA 9130, FY 2004: HTTPConnectionPool(host='dawn12.state.nv.us9130&input_fund=101&input_fiscal_year=2004', port=80): Max retries 

In [3]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

# 1. Define the target URL and parameters
url = "http://dawn12.state.nv.us"
params = {
    "pm_level1_id": "31499",
    "pm_level2_id": "95492",
    "pm_level3_id": "",
    "pm_from_date": "01/01/2020",
    "pm_to_date": "09/01/2026",
}

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

try:
    print("Fetching data from Nevada State Server...")
    response = requests.get(url, params=params, headers=headers, timeout=15)
    response.raise_for_status()

    # Try parsing tables using pandas
    try:
        tables = pd.read_html(response.text)
        if tables:
            df = tables[0]
            print(f"Success! Extracted table with {len(df)} rows.")
            df.to_csv("nevada_revenue_data.csv", index=False)
            print(df.head())
        else:
            raise ValueError("No <table> tags found.")
            
    except ValueError:
        # If pandas fails, inspect the page body to see the server error message
        print("\n[!] The server responded, but it didn't return a data table.")
        print("Printing the text response from the Nevada server below:\n" + "="*50)
        
        soup = BeautifulSoup(response.text, 'html.parser')
        # Clean up and print the visible text from the page
        server_text = soup.get_text(separator="\n").strip()
        print(server_text if server_text else response.text[:1000])
        print("="*50)

except requests.exceptions.RequestException as e:
    print(f"Network / connection error: {e}")


Fetching data from Nevada State Server...
Network / connection error: HTTPConnectionPool(host='dawn12.state.nv.us', port=80): Max retries exceeded with url: /?pm_level1_id=31499&pm_level2_id=95492&pm_level3_id=&pm_from_date=01%2F01%2F2020&pm_to_date=09%2F01%2F2026 (Caused by NewConnectionError("HTTPConnection(host='dawn12.state.nv.us', port=80): Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it"))


In [4]:
import pandas as pd
import requests

# Explicitly keep the port 7778 bound to the base URL string
url = "http://dawn12.state.nv.us"

params = {
    "pm_level1_id": "31499",
    "pm_level2_id": "95492",
    "pm_level3_id": "",
    "pm_from_date": "01/01/2020",
    "pm_to_date": "09/01/2026",
}

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

try:
    print("Attempting connection explicitly through Port 7778...")
    # Passing params directly into requests.get forces it to append safely
    response = requests.get(url, params=params, headers=headers, timeout=15)
    response.raise_for_status()
    
    tables = pd.read_html(response.text)
    print("Success! Table data retrieved.")
    
except requests.exceptions.ConnectionError as ce:
    print("\n[!] Connection Failed.")
    print("If the error still says port 7778, you are blocked by the State Firewall.")
    print(f"Details: {ce}")


Attempting connection explicitly through Port 7778...

[!] Connection Failed.
If the error still says port 7778, you are blocked by the State Firewall.
Details: HTTPConnectionPool(host='dawn12.state.nv.us', port=80): Max retries exceeded with url: /?pm_level1_id=31499&pm_level2_id=95492&pm_level3_id=&pm_from_date=01%2F01%2F2020&pm_to_date=09%2F01%2F2026 (Caused by NewConnectionError("HTTPConnection(host='dawn12.state.nv.us', port=80): Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it"))
